# Script-shift v2: summarise 50 transcripts with Qwen (all 25 windows)

Run the cells top to bottom: **Runtime → Change runtime type → T4 GPU** first.

- It needs only your existing Drive folder `MyDrive/hinglish_asr_sum`. Nothing else to upload: the cells below write the two scripts into that folder.
- It makes 2 new conditions × 25 windows = 50 summaries, about 15–20 min on a T4.
- Every summary is saved to Drive as soon as it's made. If Colab disconnects or the GPU quota runs out, reopen the notebook and run all cells again; it continues where it stopped.
- At the end it downloads `summaries_v2.csv`. Send that file back for judging-packet generation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/hinglish_asr_sum'
import os
for f in ['outputs/selected_windows.csv', 'outputs/asr_metrics_per_window_vad15.csv',
          'outputs/step2/conditions.csv', 'outputs/step2/script_shift_lexicon.csv',
          'outputs/step2/summaries.csv', 'logs/prompts_log.jsonl']:
    print('OK ' if os.path.exists(f'{ROOT}/{f}') else 'MISSING ', f)
%cd {ROOT}

In [ ]:
!pip install -q bitsandbytes accelerate
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Write the build script and make the two conditions (CPU, a few seconds)

In [ ]:
%%writefile build_script_shift_v2.py
"""Build two new input conditions (all 25 windows) for the script-shift experiment (no GPU, no API).

Usage, from inside hinglish_asr_sum/:
    python build_script_shift_v2.py

Reads only existing files. Writes:
  outputs/step2/lexicon_v2.csv          word, devanagari, source
  outputs/step2/conditions_v2.csv       same columns as conditions.csv (subset) for:
      script_shift_v2         oracle transcript with every English (Latin) token replaced
                              by its Devanagari spelling (lexicon_v2); uncovered words stay Latin
      asr:trelis+backtranslit Trelis transcript with Devanagari-spelled English words mapped
                              back to Latin. Uses a LEAVE-ONE-WINDOW-OUT lexicon (pairs harvested
                              from the other 24 windows only) plus the original Gemini lexicon.
  outputs/step2/conditions_v2_coverage.csv

Lexicon sources, in priority order:
  1. gemini   - the original script_shift_lexicon.csv (450 words)
  2. trelis   - 1:1 aligned substitutions (Latin ref word -> single Devanagari hyp word)
                harvested from Trelis output over all 25 windows; majority spelling, count >= 2
Back-transliteration safety: a Devanagari form F (for English word E) is only mapped back
if, in the OTHER windows' oracle transcripts, F occurs less often as a Devanagari token than
E occurs as a Latin token. This blocks ordinary Hindi words that collide with English
spellings (इस 'is', इन 'in', ए 'a') while keeping English words the oracle sometimes writes
in Devanagari.
"""
import collections
import difflib
import re

import pandas as pd

S2 = "outputs/step2"
sw = pd.read_csv("outputs/selected_windows.csv").set_index("window_id")
pw = pd.read_csv("outputs/asr_metrics_per_window_vad15.csv")
_g = pd.read_csv(f"{S2}/script_shift_lexicon.csv", keep_default_na=False, dtype=str)
gem = {w: d for w, d in _g.values if w and d}
cond = pd.read_csv(f"{S2}/conditions.csv")
WINDOWS = list(sw.index)

LAT = re.compile(r"^[a-z0-9']+$")
DEV = re.compile(r"^[ऀ-ॿ]+$")
NUKTA = "़"
nk = lambda s: s.replace(NUKTA, "")


def harvest(windows, model="trelis-hinglish-preview"):
    """Counter[(latin_word, devanagari_word)] from 1:1 replace ops."""
    c = collections.Counter()
    for w in windows:
        ref = sw.loc[w, "ref_norm"].split()
        hyp = str(pw[(pw.window_id == w) & (pw.model == model)].hyp_norm.iloc[0]).split()
        for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, ref, hyp, autojunk=False).get_opcodes():
            if op == "replace" and (i2 - i1) == (j2 - j1):
                for a, b in zip(ref[i1:i2], hyp[j1:j2]):
                    if LAT.match(a) and DEV.match(b):
                        c[(a, b)] += 1
    return c


def majority(counter, min_count=2):
    best = {}
    by_word = collections.defaultdict(list)
    for (a, b), n in counter.items():
        by_word[a].append((n, b))
    for a, lst in by_word.items():
        n, b = max(lst)
        if n >= min_count:
            best[a] = b
    return best


# ---------- lexicon_v2 (all windows; used for script_shift_v2) ----------
tre_all = majority(harvest(WINDOWS))
lex = {w: (d, "gemini") for w, d in gem.items()}
for w, d in tre_all.items():
    lex.setdefault(w, (d, "trelis"))
pd.DataFrame([(w, d, s) for w, (d, s) in sorted(lex.items())],
             columns=["word", "devanagari", "source"]).to_csv(f"{S2}/lexicon_v2.csv", index=False)

# token counts in each window's oracle (for the leave-one-window-out safety check)
tok_counts = {w: collections.Counter(nk(t) for t in sw.loc[w, "ref_norm"].split()) for w in WINDOWS}

rows, cov = [], []
for w in WINDOWS:
    ref = sw.loc[w, "ref_norm"].split()
    # --- script_shift_v2 ---
    out, n_en, n_cov, src = [], 0, 0, collections.Counter()
    for t in ref:
        if LAT.match(t):
            n_en += 1
            if t in lex:
                out.append(lex[t][0]); n_cov += 1; src[lex[t][1]] += 1
            else:
                out.append(t)
        else:
            out.append(t)
    rows.append(dict(window_id=w, condition="script_shift_v2", family="script_shift",
                     text=" ".join(out), n_words=len(out)))
    cov.append(dict(window_id=w, condition="script_shift_v2", n_en_tokens=n_en,
                    n_converted=n_cov, coverage=n_cov / max(1, n_en),
                    from_gemini=src["gemini"], from_trelis=src["trelis"]))

    # --- asr:trelis+backtranslit (leave-one-window-out lexicon) ---
    lowo = majority(harvest([x for x in WINDOWS if x != w]))
    lex_w = dict(gem)
    for a, b in lowo.items():
        lex_w.setdefault(a, b)
    rev = collections.defaultdict(collections.Counter)
    for a, b in lex_w.items():
        rev[nk(b)][a] += 1
    other = collections.Counter()
    for x in WINDOWS:
        if x != w:
            other.update(tok_counts[x])
    rev = {d: c.most_common(1)[0][0] for d, c in rev.items()}
    rev = {d: e for d, e in rev.items() if other[d] < other[e]}
    hyp = str(pw[(pw.window_id == w) & (pw.model == "trelis-hinglish-preview")].hyp_norm.iloc[0]).split()
    out, n_dev, n_back = [], 0, 0
    for t in hyp:
        if DEV.match(t):
            n_dev += 1
            if nk(t) in rev:
                out.append(rev[nk(t)]); n_back += 1
                continue
        out.append(t)
    rows.append(dict(window_id=w, condition="asr:trelis+backtranslit", family="script_fix",
                     text=" ".join(out), n_words=len(out)))
    cov.append(dict(window_id=w, condition="asr:trelis+backtranslit", n_dev_tokens=n_dev,
                    n_converted=n_back, coverage=n_back / max(1, n_dev)))

# sanity: Trelis text used here must equal the text used in the original run
orig = cond[cond.condition == "asr:trelis-hinglish-preview"].set_index("window_id").text
for w in WINDOWS:
    hyp = str(pw[(pw.window_id == w) & (pw.model == "trelis-hinglish-preview")].hyp_norm.iloc[0])
    assert " ".join(hyp.split()) == " ".join(str(orig[w]).split()), f"Trelis text mismatch {w}"

pd.DataFrame(rows).to_csv(f"{S2}/conditions_v2.csv", index=False)
cv = pd.DataFrame(cov)
cv.to_csv(f"{S2}/conditions_v2_coverage.csv", index=False)
print("lexicon_v2:", pd.Series([s for _, s in lex.values()]).value_counts().to_dict())
for c, g in cv.groupby("condition"):
    print(c, "converted", int(g.n_converted.sum()), "| mean per-window coverage %.2f" % g.coverage.mean())
old = cond[cond.condition == "script_shift"]
print("old script_shift rows:", len(old))


In [ ]:
!python build_script_shift_v2.py

## 2. Write the Qwen script and run it (GPU)

In [ ]:
%%writefile run_qwen_v2.py
"""Summarise the two new conditions (+ regenerated controls) with the SAME Qwen setup as Step 2.

Colab (T4) or Kaggle (T4/P100). In a notebook cell:
    !pip install -q "transformers>=4.45" bitsandbytes accelerate
    !python run_qwen_v2.py --root /content/drive/MyDrive/hinglish_asr_sum
Kaggle: upload hinglish_asr_sum as a dataset, then
    !python run_qwen_v2.py --root /kaggle/working/hinglish_asr_sum   (copy the dataset there first)

Resumable: every summary is cached to <root>/cache/qwen_v2/<key>.json the moment it is
made, so if the GPU quota cuts you off just run the same command again later.

Work: script_shift_v2 (25) then asr:trelis+backtranslit (25) = 50 summaries, about 15-20 min on a T4.
Add --controls to also regenerate oracle + Trelis (normally unnecessary).

Settings copied from logs/run_config_step2.json: Qwen/Qwen2.5-7B-Instruct @ a09a354,
bnb nf4 4-bit, fp16 compute, greedy, max_new_tokens 320, batch 4, prompt template
summary_en (checked against logs/prompts_log.jsonl at start-up). Writes outputs/step2/summaries_v2.csv at the end.
"""
import argparse
import datetime as dt
import glob
import hashlib
import json
import os

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-7B-Instruct"
REV = "a09a35458c702b33eeacc393d103063234e8bc28"
SYSTEM = "You are a helpful assistant that summarises lecture transcripts."
USER_T = (
    "The following is an automatic transcript of part of a recorded computer-software tutorial. "
    "The speaker mixes Hindi and English (Hinglish); Hindi may be written in Devanagari and English "
    "in Latin script, and words of either language may appear in either script.\n\n"
    "Write a summary in English of 120-180 words covering the main steps, concepts and facts taught. "
    "Write plain prose: no bullet points, no title.\n\n"
    "Transcript:\n<<<\n{transcript}\n>>>"
)
GEN = dict(max_new_tokens=320, do_sample=False, temperature=None, top_p=None, top_k=None)
BATCH = 4

ap = argparse.ArgumentParser()
ap.add_argument("--root", required=True)
ap.add_argument("--controls", action="store_true",
                help="also regenerate oracle + Trelis (not needed: the 400-summary rerun reproduced all 256 originals exactly)")
args = ap.parse_args()
ROOT = args.root
S2 = f"{ROOT}/outputs/step2"
CACHE = f"{ROOT}/cache/qwen_v2"
os.makedirs(CACHE, exist_ok=True)

# Check the prompt text is identical to the summary_en template logged in Step 2.
logged = [json.loads(l) for l in open(f"{ROOT}/logs/prompts_log.jsonl")]
logged = [d["prompt"] for d in logged if d.get("component") == "summary_en" and d.get("step") == "2-templates"]
assert logged and logged[0]["system"] == SYSTEM and logged[0]["user_template"] == USER_T, \
    "Prompt differs from the Step-2 template in prompts_log.jsonl - stop and check."
print("prompt template: identical to Step 2")

new = pd.read_csv(f"{S2}/conditions_v2.csv")
WINS = sorted(new.window_id.unique())
old = pd.read_csv(f"{S2}/conditions.csv")
jobs = []
for c in ["script_shift_v2", "asr:trelis+backtranslit"]:
    jobs += [(w, c, new[(new.window_id == w) & (new.condition == c)].text.iloc[0]) for w in WINS]
if args.controls:
    for c in ["oracle", "asr:trelis-hinglish-preview"]:
        jobs += [(w, c, old[(old.window_id == w) & (old.condition == c)].text.iloc[0]) for w in WINS]

key = lambda w, c: hashlib.sha256(f"{w}|{c}|en|v2".encode()).hexdigest()[:24]
todo = [j for j in jobs if not os.path.exists(f"{CACHE}/{key(j[0], j[1])}.json")]
print(f"{len(jobs) - len(todo)} cached, {len(todo)} to generate")

if todo:
    tok = AutoTokenizer.from_pretrained(MODEL, revision=REV)
    tok.padding_side = "left"
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REV, quantization_config=bnb,
                                                 device_map="auto", dtype=torch.float16)
    model.eval()
    gen = {k: v for k, v in GEN.items() if v is not None}
    for i in range(0, len(todo), BATCH):
        batch = todo[i:i + BATCH]
        prompts = [tok.apply_chat_template(
            [{"role": "system", "content": SYSTEM},
             {"role": "user", "content": USER_T.format(transcript=t)}],
            tokenize=False, add_generation_prompt=True) for _, _, t in batch]
        enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, **gen, pad_token_id=tok.eos_token_id)
        for k_, (w, c, _) in enumerate(batch):
            n_in = int(enc["attention_mask"][k_].sum())
            text = tok.decode(out[k_][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
            rec = dict(created=dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
                       model=MODEL, revision=REV, gen=str(GEN), batch=BATCH, window_id=w,
                       condition=c, out_lang="en", input_tokens=n_in, summary=text,
                       gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
                       transformers=__import__("transformers").__version__)
            json.dump(rec, open(f"{CACHE}/{key(w, c)}.json", "w"), ensure_ascii=False)
        print(f"done {min(i + BATCH, len(todo))}/{len(todo)}  ({batch[0][1]})", flush=True)
    with open(f"{ROOT}/logs/prompts_log.jsonl", "a") as f:
        f.write(json.dumps({"ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
                            "step": "2-v2", "component": "summary_en", "model": MODEL,
                            "params": {"revision": REV, "gen": GEN, "batch": BATCH,
                                       "conditions": sorted({c for _, c, _ in jobs})},
                            "prompt": {"system": SYSTEM, "user_template": USER_T}},
                           ensure_ascii=False) + "\n")

recs = [json.load(open(p)) for p in glob.glob(f"{CACHE}/*.json")]
df = pd.DataFrame(recs)
df["summary_words"] = df.summary.str.split().str.len()
df.sort_values(["condition", "window_id"]).to_csv(f"{S2}/summaries_v2.csv", index=False)
print("wrote", f"{S2}/summaries_v2.csv", df.groupby("condition").size().to_dict())

# Reproducibility check: regenerated controls vs the summaries used in the paper
saved = pd.read_csv(f"{S2}/summaries.csv")
saved = saved[saved.out_lang == "en"].set_index(["window_id", "condition"]).summary
for c in ["oracle", "asr:trelis-hinglish-preview"]:
    g = df[df.condition == c]
    if len(g):
        same = sum(str(r.summary).strip() == str(saved.get((r.window_id, c), "")).strip()
                   for r in g.itertuples())
        print(f"regenerated {c}: {same}/{len(g)} identical to the original summaries")


In [ ]:
!python run_qwen_v2.py --root {ROOT}

## 3. Download the result and send it back

In [ ]:
from google.colab import files
files.download(f'{ROOT}/outputs/step2/summaries_v2.csv')